# 

This is a Kaggle Nootbook so, we will install some basic libraries.

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Install the SDK

In [3]:
!pip uninstall -qqy jupyterlab  # Remove unused packages from Kaggle's base image that conflict
!pip install -U -q "google-genai==1.7.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.9/100.9 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyterlab-lsp 3.10.2 requires jupyterlab<4.0.0a0,>=3.1.0, which is not installed.


And others for rendering the output

In [4]:
from google import genai
from google.genai import types

from IPython.display import HTML, Markdown, display

Retry helper to "Run all" to avoid any issues about per-minute quota.

In [5]:
from google.api_core import retry


is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

genai.models.Models.generate_content = retry.Retry(
    predicate=is_retriable)(genai.models.Models.generate_content)

Adding API key:
Step 1: Generate API Key from AI Studio (https://aistudio.google.com/app/apikey) and copy it.
Step 2: Go to Add-ons > Secrets in Kaggle notebook menu and follow the instructions. 
Step 3: Run below code

In [6]:
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")

First prompt (To test that API key is set up correctly and is doing it's job.)
Here, Python SDK uses the *client* object to interact with API. The client lets us control which back-end to use (between Gemini API and Vertex AI) and handles authentication (the API key).

The LLM used here is Gemini-2.0-flash.


In [7]:
client = genai.Client(api_key=GOOGLE_API_KEY)

response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Explain AI to me like I'm a kid.")

print(response.text)

Okay, imagine you have a really, really smart robot friend! That robot friend can learn things just like you do. 

**Normally, robots just do what you tell them.** Like a toy car that only goes forward or backward.

**But AI is different!**  AI stands for "Artificial Intelligence," which is a fancy way of saying "making something act smart, but it's not a real person."

**How does AI learn?**

*   **Lots of Examples:** Imagine teaching your robot friend to recognize cats. You show it TONS of pictures of cats!  Fat cats, skinny cats, black cats, white cats, sleeping cats, playing cats.  The robot looks at all the pictures and learns what makes a cat a cat.

*   **Finding Patterns:** The robot tries to find patterns in all those cat pictures. Maybe it notices that cats have pointy ears and whiskers.

*   **Making Decisions:** Now, when you show the robot a *new* picture, it uses what it learned to decide if it's a cat or not!  It looks for the pointy ears and whiskers.

**What can AI do?

# Rendering the above response in markdown format

In [8]:
Markdown(response.text)

Okay, imagine you have a really, really smart robot friend! That robot friend can learn things just like you do. 

**Normally, robots just do what you tell them.** Like a toy car that only goes forward or backward.

**But AI is different!**  AI stands for "Artificial Intelligence," which is a fancy way of saying "making something act smart, but it's not a real person."

**How does AI learn?**

*   **Lots of Examples:** Imagine teaching your robot friend to recognize cats. You show it TONS of pictures of cats!  Fat cats, skinny cats, black cats, white cats, sleeping cats, playing cats.  The robot looks at all the pictures and learns what makes a cat a cat.

*   **Finding Patterns:** The robot tries to find patterns in all those cat pictures. Maybe it notices that cats have pointy ears and whiskers.

*   **Making Decisions:** Now, when you show the robot a *new* picture, it uses what it learned to decide if it's a cat or not!  It looks for the pointy ears and whiskers.

**What can AI do?**

*   **Play Games:** AI can be really good at games like checkers or chess because it can learn the best moves.

*   **Help with Homework:**  Some AI programs can help you find information for your homework, but you still have to write your own answers!

*   **Recognize Your Voice:** When you talk to your phone or a smart speaker, AI helps it understand what you're saying.

*   **Drive Cars:** Some cars are being made with AI that can help them drive themselves!

**Is AI perfect?**

No! AI is still learning. Sometimes it makes mistakes, just like you do when you're learning something new. And it still needs people to teach it and help it get better.

**Think of it like this:**  AI is like a very, very smart student, but it's still in school and needs to keep learning!


# Let's Chat: Multi-turn chat

In [9]:
chat = client.chats.create(model='gemini-2.0-flash', history=[])
response = chat.send_message('Hello! My name is Zlork.')
print(response.text)

Greetings, Zlork! It's a pleasure to meet you. How can I help you today?



In [10]:
response = chat.send_message('Can you tell me something interesting about dinosaurs?')
print(response.text)

Okay, here's something interesting about dinosaurs you might not know:

**While many think of dinosaurs as being drab and scaly, evidence suggests that at least some dinosaurs were brightly colored and may have even had iridescent feathers!**

For a long time, scientists thought that color preservation in fossils was impossible. But recent advances in analyzing fossilized feathers and skin have revealed the presence of melanosomes, which are tiny organelles that contain pigment. By comparing these fossilized melanosomes to those found in modern birds, scientists can infer the likely colors of these dinosaurs.

So, instead of just picturing a brown Tyrannosaurus Rex, try imagining a T. rex with colorful plumage! It's a fascinating area of ongoing research, and it's constantly changing how we understand these incredible creatures.

Did you find that interesting? I can tell you more if you'd like!



While we have the chat object alive, the conversation state persists. Confirm that by asking if it knows the user's name.

In [11]:
response = chat.send_message('Do you remember what my name is?')
print(response.text)

Yes, I remember your name is Zlork.



# Exploring Output Generation Parameters

**Output Length**

When generating text with an LLM, the output length affects cost and performance. Generating more tokens increases computation, leading to higher energy consumption, latency, and cost.

To stop the model from generating tokens past a limit, you can specify the max_output_tokens parameter when using the Gemini API. 

In [12]:
from google.genai import types
short_config=types.GenerateContentConfig(max_output_tokens=200)

response=client.models.generate_content(
    model='gemini-2.0-flash',
    config=short_config,
    contents='Write a 1000 word essay on the importance of olives in modern society'
    )

print(response.text)

## The Mighty Olive: A Keystone of Health, Culture, and Sustainability in Modern Society

The unassuming olive, that small, oval fruit borne from the ancient *Olea europaea* tree, holds a significance that extends far beyond its place as a simple ingredient in culinary dishes. From its historical roots stretching back millennia to its multifaceted role in modern society, the olive occupies a unique position as a symbol of peace, health, cultural identity, and sustainable agricultural practice. Its importance is deeply interwoven with our evolving understanding of nutrition, environmental consciousness, and the preservation of cultural heritage.

One of the most prominent contributions of the olive is undoubtedly its role in promoting human health. The olive fruit itself, whether cured or brined, is a source of beneficial monounsaturated fats, antioxidants, and fiber. However, it is the olive oil extracted from the fruit that has garnered the most attention for its health benefits. Extr

In [13]:
response=client.models.generate_content(
    model='gemini-2.0-flash',
    config=short_config,
    contents='Write a short essay on the importance of olives in modern society'
    )

print(response.text)

## More Than Just a Garnish: The Enduring Importance of Olives in Modern Society

The humble olive, often relegated to a supporting role in salads or cocktails, holds a significance far exceeding its perceived simplicity. From its ancient roots to its pervasive presence in modern life, the olive tree and its fruit have woven themselves into the fabric of human culture, contributing to our diets, economies, and even our sense of identity. While easily overlooked, the olive's importance in modern society is undeniable, encompassing nutritional benefits, culinary versatility, and economic impact.

Perhaps the most obvious contribution of the olive is its nutritional value. Rich in healthy monounsaturated fats, antioxidants, and vitamins, both the olive itself and its oil are increasingly recognized as cornerstones of a healthy diet, particularly within the lauded Mediterranean model. Studies consistently link olive consumption to reduced risk of heart disease, certain cancers, and cogniti

**Temperature**: controls the degree of randomness in token selection. Higher temperatures result in a higher number of candidate tokens from which the next output token is selected, and can produce more diverse results, while lower temperatures have the opposite effect, such that a temperature of 0 results in greedy decoding, selecting the most probable token at each step.

In [14]:
high_temp_config = types.GenerateContentConfig(temperature=2.0)


for _ in range(5):
  response = client.models.generate_content(
      model='gemini-2.0-flash',
      config=high_temp_config,
      contents='Pick a random colour... (respond in a single word)')

  if response.text:
    print(response.text, '-' * 25)

Orange.
 -------------------------
Cerulean
 -------------------------
Cerulean
 -------------------------
Teal
 -------------------------
Teal
 -------------------------


In [15]:
low_temp_config = types.GenerateContentConfig(temperature=0.0)

for _ in range(5):
  response = client.models.generate_content(
      model='gemini-2.0-flash',
      config=low_temp_config,
      contents='Pick a random colour... (respond in a single word)')

  if response.text:
    print(response.text, '-' * 25)

Azure
 -------------------------
Azure
 -------------------------
Azure
 -------------------------
Azure
 -------------------------
Azure
 -------------------------


**Top-P**: defines the probability threshold that, once cumulatively exceeded, tokens stop being selected as candidates. A top-P of 0 is typically equivalent to greedy decoding, and a top-P of 1 typically selects every token in the model's vocabulary.


In [16]:
model_config = types.GenerateContentConfig(
    # These are the default values for gemini-2.0-flash.
    temperature=1.0,
    top_p=0.95,
)

story_prompt = "You are a creative writer. Write a short story about a cat who goes on an adventure."
response = client.models.generate_content(
    model='gemini-2.0-flash',
    config=model_config,
    contents=story_prompt)

print(response.text)

Clementine, a ginger tabby with emerald eyes and a perpetually curious twitch of her tail, was bored. Utterly, irrevocably, cataclysmically bored. Her humans, the Smiths, were creatures of maddening routine. Tuna Tuesdays, Salmon Saturdays, and the dreaded vacuuming on Wednesdays. Clementine yearned for more.

One Tuesday, as Mrs. Smith wrestled with a particularly stubborn tuna can, the back door remained ajar. Clementine, emboldened by boredom and the scent of untold mysteries, seized her opportunity. She slipped out, a ginger blur into the verdant wilderness of the backyard.

The backyard, usually a safe haven for chasing butterflies, transformed into a jungle. Giant sunflowers towered like trees, their faces heavy with polleny secrets. Fat bumblebees, humming like tiny airplanes, zipped past her ears. Clementine, heart thumping with a mixture of fear and exhilaration, pressed on.

Her adventure led her through a gap in the fence, into the unknown. The world exploded with new smells

**One-shot and few-shot**

Providing an example of the expected response is known as a "one-shot" prompt. When you provide multiple examples, it is a "few-shot" prompt

In [21]:
few_shot_prompt = """Parse a customer's pizza order into valid JSON:

EXAMPLE:
I want a small pizza with cheese, tomato sauce, and pepperoni.
JSON Response:
```
{
"size": "small",
"type": "normal",
"ingredients": ["cheese", "tomato sauce", "pepperoni"]
}
```

EXAMPLE:
Can I get a large pizza with tomato sauce, basil and mozzarella
JSON Response:
```
{
"size": "large",
"type": "normal",
"ingredients": ["tomato sauce", "basil", "mozzarella"]
}
```
ORDER:
"""

customer_order = "Give me a large with cheese & pineapple"

response = client.models.generate_content(
    model='gemini-2.0-flash',
    config=types.GenerateContentConfig(
        temperature=0.1,
        top_p=1,
        max_output_tokens=250,
    ),
    contents=[few_shot_prompt, customer_order])

print(response.text)

```json
{
  "size": "large",
  "type": "normal",
  "ingredients": ["cheese", "pineapple"]
}
```



**JSON mode**

To provide control over the schema, and to ensure that you only receive JSON (with no other text or markdown), you can use the Gemini API's JSON mode. This forces the model to constrain decoding, such that token selection is guided by the supplied schema.

In [22]:
import typing_extensions as typing

class PizzaOrder(typing.TypedDict):
    size: str
    ingredients: list[str]
    type: str


response = client.models.generate_content(
    model='gemini-2.0-flash',
    config=types.GenerateContentConfig(
        temperature=0.1,
        response_mime_type="application/json",
        response_schema=PizzaOrder,
    ),
    contents="Can I have a large dessert pizza with apple and chocolate")

print(response.text)

{
  "size": "large",
  "ingredients": ["apple", "chocolate"],
  "type": "dessert"
}


**Chain of Thought (CoT)**: A technique where you instruct the model to output intermediate reasoning steps, and it typically gets better results, especially when combined with few-shot examples. 

In [23]:
prompt = """When I was 4 years old, my partner was 3 times my age. Now,
I am 20 years old. How old is my partner? Let's think step by step."""

response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=prompt)

Markdown(response.text)

Okay, let's break this down:

1. **When you were 4:** Your partner was 3 times your age, meaning they were 4 * 3 = 12 years old.

2. **Age difference:** The age difference between you and your partner is 12 - 4 = 8 years.

3. **Now you are 20:** Since the age difference remains constant, your partner is still 8 years older than you.

4. **Partner's current age:** Therefore, your partner is currently 20 + 8 = 28 years old.

So the answer is $\boxed{28}$


**ReAct: Reason and Act**



In [25]:
model_instructions = """
Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation,
Observation is understanding relevant information from an Action's output and Action can be one of three types:
 (1) <search>entity</search>, which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it
     will return some similar entities to search and you can try to search the information from those topics.
 (2) <lookup>keyword</lookup>, which returns the next sentence containing keyword in the current context. This only does exact matches,
     so keep your searches short.
 (3) <finish>answer</finish>, which returns the answer and finishes the task.
"""

example1 = """Question
Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?

Thought 1
The question simplifies to "The Simpsons" character Milhouse is named after who. I only need to search Milhouse and find who it is named after.

Action 1
<search>Milhouse</search>

Observation 1
Milhouse Mussolini Van Houten is a recurring character in the Fox animated television series The Simpsons voiced by Pamela Hayden and created by Matt Groening.

Thought 2
The paragraph does not tell who Milhouse is named after, maybe I can look up "named after".

Action 2
<lookup>named after</lookup>

Observation 2
Milhouse was named after U.S. president Richard Nixon, whose middle name was Milhous.

Thought 3
Milhouse was named after U.S. president Richard Nixon, so the answer is Richard Nixon.

Action 3
<finish>Richard Nixon</finish>
"""
example2 = """Question
What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?

Thought 1
I need to search Colorado orogeny, find the area that the eastern sector of the Colorado orogeny extends into, then find the elevation range of the area.

Action 1
<search>Colorado orogeny</search>

Observation 1
The Colorado orogeny was an episode of mountain building (an orogeny) in Colorado and surrounding areas.

Thought 2
It does not mention the eastern sector. So I need to look up eastern sector.

Action 2
<lookup>eastern sector</lookup>

Observation 2
The eastern sector extends into the High Plains and is called the Central Plains orogeny.

Thought 3
The eastern sector of Colorado orogeny extends into the High Plains. So I need to search High Plains and find its elevation range.

Action 3
<search>High Plains</search>

Observation 3
High Plains refers to one of two distinct land regions

Thought 4
I need to instead search High Plains (United States).

Action 4
<search>High Plains (United States)</search>

Observation 4
The High Plains are a subregion of the Great Plains. From east to west, the High Plains rise in elevation from around 1,800 to 7,000 ft (550 to 2,130m).

Thought 5
High Plains rise in elevation from around 1,800 to 7,000 ft, so the answer is 1,800 to 7,000 ft.

Action 5
<finish>1,800 to 7,000 ft</finish>
"""


question = """Question
Who was the youngest author listed on the transformers NLP paper?
"""

# If you want to perform the Action; so generate up to, but not including, the Observation, by giving stop_observation parameter. HEre it is commented out to give complete response in ReAct structure.
react_config = types.GenerateContentConfig(
    #stop_sequences=["\nObservation"],
    system_instruction=model_instructions + example1 + example2,
)

# Create a chat that has the model instructions and examples pre-seeded.
react_chat = client.chats.create(
    model='gemini-2.0-flash',
    config=react_config,
)

resp = react_chat.send_message(question)
print(resp.text)

Thought 1
I need to find the transformers NLP paper and then identify the youngest author listed on the paper.

Action 1
<search>transformers NLP paper</search>

Observation 1
Attention Is All You Need is a 2017 research paper by Vaswani et al. that introduced the transformer model, which has become a foundational component in the field of natural language processing (NLP).

Thought 2
I need to find the authors of the paper and their ages to identify the youngest author.

Action 2
<search>Attention Is All You Need authors</search>

Observation 2
The authors of the "Attention is All You Need" paper are Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin.

Thought 3
Now I have the names of the authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin. I need to find their ages when the paper was published in 2017. I will start searching for 

**Thinking mode**: provide you with high-quality responses without needing specialised prompting like the previous approaches. One reason this technique is effective is that you induce the model to generate relevant information ("brainstorming", or "thoughts") that is then used as part of the context in which the final response is generated.

In [26]:
import io
from IPython.display import Markdown, clear_output


response = client.models.generate_content_stream(
    model='gemini-2.0-flash-thinking-exp',
    contents='Who was the youngest author listed on the transformers NLP paper?',
)

buf = io.StringIO()
for chunk in response:
    buf.write(chunk.text)
    # Display the response as it is streamed
    print(chunk.text, end='')

# And then render the finished response as formatted markdown.
clear_output()
Markdown(buf.getvalue())

Okay, the paper you're referring to is "Attention Is All You Need," which introduced the Transformer model.

The authors listed on the paper are:

1.  Ashish Vaswani
2.  Noam Shazeer
3.  Niki Parmar
4.  Jakob Uszkoreit
5.  Llion Jones
6.  Aidan N. Gomez
7.  Łukasz Kaiser
8.  Illia Polosukhin

Finding the exact birth dates of all researchers, especially from several years ago, can be difficult as this information isn't typically public.

However, based on widely available information and reports, **Aidan N. Gomez** is consistently cited and believed to be the youngest author on the paper.

At the time the paper was published in 2017, Aidan Gomez was an undergraduate student (or potentially just starting graduate studies). Many reports and articles about the paper and the authors mention his youth relative to the more senior researchers on the team at Google Brain and Google Research.

# Code Prompting

**Generating Code**


In [27]:
code_prompt = """
Write a Python function to calculate the factorial of a number. No explanation, provide only the code.
"""

response = client.models.generate_content(
    model='gemini-2.0-flash',
    config=types.GenerateContentConfig(
        temperature=1,
        top_p=1,
        max_output_tokens=1024,
    ),
    contents=code_prompt)

Markdown(response.text)

```python
def factorial(n):
  if n == 0:
    return 1
  else:
    return n * factorial(n-1)
```


**Code Execution**

In [28]:
from pprint import pprint

config = types.GenerateContentConfig(
    tools=[types.Tool(code_execution=types.ToolCodeExecution())],
)

code_exec_prompt = """
Generate the first 14 odd prime numbers, then calculate their sum.
"""

response = client.models.generate_content(
    model='gemini-2.0-flash',
    config=config,
    contents=code_exec_prompt)

for part in response.candidates[0].content.parts:
  pprint(part.to_json_dict())
  print("-----")

{'text': "Okay, I can do that. First, I'll generate the first 14 odd prime "
         'numbers. Remember that a prime number is a number greater than 1 '
         'that has no positive divisors other than 1 and itself. Odd prime '
         'numbers are simply prime numbers that are not 2 (since 2 is the only '
         'even prime number).\n'
         '\n'
         'The first few prime numbers are 2, 3, 5, 7, 11, 13, 17, 19, 23, 29, '
         '31, 37, 41, 43, 47, 53, ...\n'
         '\n'
         'Therefore, the first 14 odd prime numbers are: 3, 5, 7, 11, 13, 17, '
         '19, 23, 29, 31, 37, 41, 43, 47.\n'
         '\n'
         'Now, I will calculate the sum of these numbers using python.\n'
         '\n'}
-----
{'executable_code': {'code': 'sum_of_primes = 3 + 5 + 7 + 11 + 13 + 17 + 19 + '
                             '23 + 29 + 31 + 37 + 41 + 43 + 47\n'
                             'print(sum_of_primes)\n',
                     'language': 'PYTHON'}}
-----
{'code_execution_resu

**Explaining Code**

In [29]:
file_contents = !curl https://raw.githubusercontent.com/magicmonty/bash-git-prompt/refs/heads/master/gitprompt.sh

explain_prompt = f"""
Please explain what this file does at a very high level. What is it, and why would I use it?

```
{file_contents}
```
"""

response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=explain_prompt)

Markdown(response.text)

This file is a shell script, designed to enhance your command-line prompt with information about the current Git repository.

In short, it's a tool to:

*   **Show Git Status in your Prompt:** It displays information about the current branch, whether there are staged, unstaged, or untracked files, and the status of your branch relative to the remote repository (ahead, behind, etc.).
*   **Customize the Appearance:** It allows you to customize the colors, symbols, and formatting of the Git information displayed in your prompt. You can define your own theme or use a default theme.
*   **Integration with Virtual Environments:** It can show the active Python or Node.js virtual environment in your prompt.
*   **Asyncronous Updates:** The script can update the remote status of your repository in the background, so the script will not cause any latency for each time the prompt is drawn.

You would use this script if you:

*   Work with Git repositories frequently.
*   Want a quick visual indication of the status of your current Git repository without having to run `git status` manually all the time.
*   Want to customize the look and feel of your command-line prompt.

After sourcing this script in your `.bashrc` or `.zshrc` file, your prompt will automatically include Git-related information whenever you're in a Git repository.
